# Parts 3–4 · 01 Download

The only notebook in parts 3–4 that talks to WRDS. It saves raw pulls to `cache/` as parquet files, one per table (and per year for the big ones), and logs each pull in `cache/manifest.csv`.

**Rerun-safe:** anything already saved is skipped. If the kernel stops, rerun the folder and connect cells, then the pull cell. Only missing years are pulled.

**Before running:** put `common.py` and `download.py` in `3_4_surprise_returns/code/`. Nothing here uses Mahaut's files.

`cache/` holds WRDS data: it stays in the team's private storage and is never part of the submission.

In [ ]:
%pip install -q wrds pandas_market_calendars pyarrow


In [ ]:
import importlib
import os
import sys
from pathlib import Path

_start = Path(os.environ.get("S08_ROOT", Path.cwd())).expanduser().resolve()
REPO_ROOT = next((p for p in (_start, *_start.parents)
                  if (p / "scripts" / "project_paths.py").is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Open the notebook inside S08, or set S08_ROOT to the checkout path.")
sys.path.insert(0, str(REPO_ROOT / "scripts"))
import project_paths
importlib.reload(project_paths)
from project_paths import (REPO_ROOT, TEAM_DIR, SCRIPTS, NOV2_DATA,
                           PROJECT_DIR, STEP5_DIR, PART34_CODE, STEP5_CODE)

# Notebook paths.


os.environ["PROJECT_DIR"] = str(PROJECT_DIR)
sys.path.insert(0, str(PART34_CODE))
missing = [f for f in ["common.py", "download.py"] if not (PART34_CODE / f).exists()]
assert not missing, f"Missing code in {PROJECT_DIR / 'code'}: {missing}"
print("project folder:", PROJECT_DIR)

In [ ]:
import wrds

WRDS_USER = input("WRDS username: ").strip()
db = wrds.Connection(wrds_username=WRDS_USER)   # answer n to the .pgpass question

## Core pulls

What the checkpoint needs:
- **Links:** I/B/E/S to CRSP, Compustat to CRSP, EDGAR CIK to Compustat.
- **I/B/E/S:** quarterly EPS actuals and individual estimates (unadjusted).
- **CRSP daily:** prices, opens, returns, adjustment factors and delistings.
- **Market data:** daily Fama-French factors and NYSE size breakpoints.

All years run 2005–2025. Rerunning this cell pulls only what is missing, for example the stopped estimates and I/B/E/S surprise summary added after the first run. The big pulls go one year at a time, and the printed timing of the first year tells you roughly how long the rest will take. If a line says `FAILED`, copy it to me; the other pulls carry on.

In [ ]:
import importlib
import download as d
importlib.reload(d)

result = d.run(db, d.CORE, user=WRDS_USER)
print()
print(result["status"].value_counts().to_string())

## Sanity checks

Quick look at what came back. Copy this cell's output to me.

In [ ]:
import pandas as pd

C = PROJECT_DIR / "cache"


def load(name):
    files = [f for f in sorted((C / name).glob("*.parquet")) if ".partial" not in f.name]
    return pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)


act = load("ibes_actuals")
t = act["anntims"].astype("string")
print(f"I/B/E/S quarterly EPS actuals: {len(act):,} | tickers: {act.ticker.nunique():,}")
print("share with a usable announcement time:", round((t.notna() & t.ne("00:00:00")).mean(), 3))
print("currencies:", act["curr_act"].value_counts().head(4).to_dict())
print("per year:", act.groupby(act["anndats"].dt.year).size().to_dict())

det = pd.read_parquet(C / "ibes_detail" / "2015.parquet")
print(f"\nestimates made in 2015: {len(det):,} | fpi: {det.fpi.value_counts().to_dict()}")
print("currencies:", det["curr"].value_counts().head(4).to_dict(),
      "| basic/diluted flag:", det["pdf"].value_counts().to_dict())

px = pd.read_parquet(C / "crsp_daily" / "2015.parquet")
print(f"\nCRSP 2015: {len(px):,} rows | {px.permno.nunique():,} stocks | {px.date.nunique()} days")
print("share of rows with an open price:", round(px["dlyopen"].notna().mean(), 3),
      "| with a return:", round(px["dlyret"].notna().mean(), 3))

ff = pd.read_parquet(C / "ff_daily.parquet")
print(f"\nFama-French daily: {ff.date.min().date()} to {ff.date.max().date()}")

bp = load("nyse_breakpoints")
print(f"NYSE breakpoints: {len(bp)} month-ends, {bp.date.min().date()} to {bp.date.max().date()}")

## Later: wider baseline and GAAP check (optional now)

Revenue actuals and estimates, company guidance, and Compustat GAAP EPS. They aren't needed for the checkpoint, so run this whenever there's time. The `peek` line only records the columns of tables we haven't used yet (stopped estimates, the I/B/E/S surprise summary); it pulls no data.

In [ ]:
schemas = d.peek(db)

In [ ]:
# Why did revenue come back empty? Row counts by measure in the unadjusted and
# adjusted tables. Copy this output to me before rerunning the revenue pulls.
d.measure_counts(db)

In [ ]:
result_later = d.run(db, d.LATER, user=WRDS_USER)
print(result_later["status"].value_counts().to_string())

## What's in the cache

In [ ]:
print(d.summary().to_string())

In [12]:
db.close()